<a href="https://colab.research.google.com/github/sonaliliyanahetti/Final-Year-Research-Malaria-Detection-Models/blob/main/parasite_inflator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Paths and setup
import os, random, math, shutil, pathlib

DATASET_PATH = "/content/drive/MyDrive/Research/archive/cell_images/cell_images"  # <-- adjust if needed
WORK_DIR = "/content/malaria_subset_4k"
os.makedirs(WORK_DIR, exist_ok=True)

classes = ["Parasitized", "Uninfected"]

# 3. Create 2000-image subset (1000 per class)
files_by_class = {}
SAMPLES_PER_CLASS = 1000  # <-- requested
for cls in classes:
    cls_dir = pathlib.Path(DATASET_PATH) / cls
    files = [str(p) for p in cls_dir.glob("*") if p.suffix.lower() in [".png", ".jpg", ".jpeg"]]
    random.shuffle(files)
    if len(files) < SAMPLES_PER_CLASS:
        raise ValueError(f"Not enough images in {cls}: found {len(files)}, need {SAMPLES_PER_CLASS}")
    files_by_class[cls] = files[:SAMPLES_PER_CLASS]

# Split 70/15/15
splits = {"train": 0.7, "val": 0.15, "test": 0.15}
for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(WORK_DIR, split, cls), exist_ok=True)

for cls in classes:
    files = files_by_class[cls]
    n = len(files)
    n_train = math.floor(n * splits["train"])     # 1400 per class
    n_val   = math.floor(n * splits["val"])       # 300 per class
    train_files = files[:n_train]
    val_files   = files[n_train:n_train+n_val]
    test_files  = files[n_train+n_val:]

    for f in train_files:
        shutil.copy(f, os.path.join(WORK_DIR, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(WORK_DIR, "val", cls, os.path.basename(f)))
    for f in test_files:
        shutil.copy(f, os.path.join(WORK_DIR, "test", cls, os.path.basename(f)))

print("Subset created at:", WORK_DIR)


In [ ]:
# 4. tf.data pipeline
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # reduce to 16 if you hit GPU memory limits
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(split):
    ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(WORK_DIR, split),
        labels="inferred",
        label_mode="binary",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=(split == "train"),
    )
    return ds

train_ds = make_ds("train")
val_ds   = make_ds("val")
test_ds  = make_ds("test")

# Augmentation + preprocessing
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
], name="augment")

# ================= Parasite Inflator (OpenCV wrapped for TF) =================
import cv2
import numpy as np

# OpenCV enhancement function
def parasite_inflator_tf(image, label):
    image = image.numpy().astype(np.uint8)

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    # Sharpen
    kernel = np.array([[0,-1,0],
                       [-1,5,-1],
                       [0,-1,0]])
    sharp = cv2.filter2D(enhanced, -1, kernel)

    # Inflate features (dilation)
    kernel = np.ones((3,3), np.uint8)
    inflated = cv2.dilate(sharp, kernel, iterations=1)

    # Convert back to 3-channel RGB
    inflated = cv2.cvtColor(inflated, cv2.COLOR_GRAY2RGB)

    return inflated, label


# TensorFlow wrapper
def tf_inflator(x, y):
    x, y = tf.py_function(
        func=parasite_inflator_tf,
        inp=[x, y],
        Tout=[tf.uint8, tf.float32]
    )

    x.set_shape([224, 224, 3])
    y.set_shape([])

    return x, y
# ============================================================================


def prepare(ds, training=False):

    # Apply Parasite Inflator FIRST
    ds = ds.map(tf_inflator, num_parallel_calls=AUTOTUNE)

    # Use MobileNetV2 preprocessing (scale to [-1, 1])—works fine for ViT too
    ds = ds.map(lambda x, y: (tf.keras.applications.mobilenet_v2.preprocess_input(x), y),
                num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(4096)
    return ds.prefetch(AUTOTUNE)

train_ds = prepare(train_ds, training=True)
val_ds   = prepare(val_ds, training=False)
test_ds  = prepare(test_ds, training=False)